# NLP Models and Functions Analysis

This notebook summarizes the data and shows small, reproducible checks for the NLP models
and functions in the tutor pipeline. It is designed to run inside the repo without
requiring the full training datasets.

Some cells use spaCy models or transformer embeddings. Those cells handle missing
dependencies gracefully and can be skipped if models are not installed.


## Setup


In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if (ROOT / "nlp_tutor").exists():
    PROJECT_ROOT = ROOT
else:
    PROJECT_ROOT = ROOT.parent.parent

PROJECT_ROOT


PosixPath('/home/joels/PycharmProjects/nlp-adaptive-tutor')

## Lesson bank data overview
The lesson bank is the curated dataset used for prompts and targets.


In [2]:
lesson_path = PROJECT_ROOT / "nlp_tutor" / "resources" / "lesson_bank.csv"
lesson_df = pd.read_csv(lesson_path)
lesson_df.head()


,lesson_id,item_id,topic,prompt_en,target_es,target_en,target_pl,gloss_en
0,1,1,education,Say you study NLP at university.,Estudio NLP en la universidad hoy.,I study NLP at the university.,Studiuję NLP na uniwersytecie w tym roku.,I study NLP at the university.
1,1,2,education,Say you work on your thesis.,Trabajo en mi tesis este año.,I work on my thesis this year.,Pracuję nad moją tezą w tym roku.,I work on my thesis this year.
2,1,3,education,Say you have an NLP assignment.,Tengo un trabajo de NLP mañana.,I have an NLP assignment due tomorrow.,Mam zadanie z NLP na jutro.,I have an NLP assignment due tomorrow.
3,2,1,daily,Ask what time it is now.,¿Qué hora es ahora mismo exactamente?,What time is it right now?,Jaka jest teraz dokładnie godzina u ciebie?,What time is it right now?
4,2,2,daily,Say you get up at seven.,Me levanto a las siete cada día.,I get up at seven each day.,Wstaję o siódmej każdego dnia rano.,I get up at seven each day.


In [3]:
lesson_df.shape, lesson_df.columns.tolist()


((12, 8),
 ['lesson_id',
  'item_id',
  'topic',
  'prompt_en',
  'target_es',
  'target_en',
  'target_pl',
  'gloss_en'])

In [4]:
lesson_df["topic"].value_counts()


topic
education    3
daily        3
travel       3
food         3
Name: count, dtype: int64

## Language identification (char TF-IDF + logistic regression)
We load the baseline model shipped with the repo and run a few sample predictions.


In [5]:
from nlp_tutor.classification.lang_detect_baseline import load_model, predict_language

lang_model = load_model()

samples = [
    "Me llamo Joel y estudio informatica.",
    "I like language models and chatbots.",
    "To jest bardzo krotkie zdanie.",
    "hola gracias",
]

rows = []
for text in samples:
    out = predict_language(text, model=lang_model, top_k=3)
    for label, score in out["top_k"]:
        rows.append({"text": text, "label": label, "score": round(score, 4)})

pd.DataFrame(rows)


,text,label,score
0,Me llamo Joel y estudio informatica.,Spanish,0.6099
1,Me llamo Joel y estudio informatica.,English,0.2393
2,Me llamo Joel y estudio informatica.,Polish,0.1508
3,I like language models and chatbots.,English,0.6063
4,I like language models and chatbots.,Spanish,0.2148
5,I like language models and chatbots.,Polish,0.1789
6,To jest bardzo krotkie zdanie.,Polish,0.7663
7,To jest bardzo krotkie zdanie.,English,0.1267
8,To jest bardzo krotkie zdanie.,Spanish,0.1069
9,hola gracias,Spanish,0.4307


Model results from the README (WiLI 2018 sample):
- Baseline char TF-IDF + logreg: Accuracy 0.9927, Macro F1 0.9927.
- Char BiLSTM: Accuracy 0.9440, Macro F1 0.9437.


## Semantic similarity (TF-IDF baseline)


In [6]:
from nlp_tutor.languages import Lang
from nlp_tutor.semantics import SemanticScorer

spanish_targets = lesson_df["target_es"].dropna().astype(str).tolist()

scorer = SemanticScorer("tfidf")
scorer.fit(spanish_targets, lang=Lang.ES)

query = "Me llamo Joel y estudio informatica."
nearest = scorer.nearest(query, k=5)

pd.DataFrame(nearest, columns=["target_es", "tfidf_cosine"])


,target_es,tfidf_cosine
0,estudio nlp en la universidad hoy.,0.239986
1,me levanto a las siete cada día.,0.213201
2,tengo un trabajo de nlp mañana.,0.000000
3,trabajo en mi tesis este año.,0.000000
4,¿qué hora es ahora mismo exactamente?,0.000000


In [7]:
target = spanish_targets[0]
scorer.score_pair(query, target, lang=Lang.ES)


SimilarityResult(backend='tfidf', score=0.05355095314996731, interpretation='Meaning does not match the target well.')

Semantic sanity check from the README (lesson bank):
- TF-IDF: mean positive 1.0000, mean negative 0.0133, pairwise accuracy 1.00.
- Transformer: mean positive 1.0000, mean negative 0.2634, pairwise accuracy 1.00.


## Core functions in the pipeline
These checks exercise preprocessing, policy logic, syntax, NER, and fluency.


In [8]:
from nlp_tutor import preprocessing as prep
from nlp_tutor.dialogue_policy import choose_action

sample_text = "  Me llamo Joel y estudio informatica.  "
prep.normalise(sample_text)

choose_action(
    expected_lang="ES",
    detected_lang="ES",
    lang_conf_ok=True,
    syntax_issues=[],
    semantic_score=0.92,
    fluency_band=None,
)


TutorAction(code='GOOD', message='Good job. That matches the meaning.', severity='info')

In [9]:
from nlp_tutor.languages import Lang
from nlp_tutor.syntax import analyse_syntax
from nlp_tutor.ner import extract_ner

text = "Me llamo Joel y estudio informatica."

try:
    syntax = analyse_syntax(text=text, lang=Lang.ES)
    syntax.features, syntax.issues
except Exception as exc:
    print("Syntax analysis skipped:", exc)

try:
    ner = extract_ner(lang=Lang.ES, text=text)
    ner.entities, ner.noun_phrases
except Exception as exc:
    print("NER skipped:", exc)


In [10]:
from nlp_tutor.fluency import FluencyScorer
from nlp_tutor.languages import Lang

toy_sents = [
    ["me", "llamo", "joel", "y", "estudio", "informatica"],
    ["hola", "como", "estas"],
    ["me", "gusta", "el", "nlp"],
]

fluency = FluencyScorer(order=2)

try:
    fluency.train_from_tokenized(Lang.ES, toy_sents, calibrate=True)
    fluency.score_text("me llamo joel", lang=Lang.ES)
except Exception as exc:
    print("Fluency demo skipped:", exc)
